# MAPK Base-Editing Screens — Master Analysis

Plotting primitives come from [`be_scan`](https://github.com/liaulab/be-scan). Everything specific to this manuscript lives in `code/`. Screen data is read from `TableS1-ScreenData.xlsx`. Reference data from `TableS3-cBioPortal-Annotations.xlsx`, `TableS4-DMS-Annotations.xlsx`, `TableS5-PocketDistances-Annotations.xlsx`. All figures are written to `Outputs/`.

Section 0 provisions the environment from scratch, so no pre-existing conda environment is needed. Sections 4 onward are independent of each other and can be run selectively once sections 0–3 have run.

See `README.md` for the figure inventory. Colab Notebook is recommended for reproducing figures.

To reproduce the figures, you need the ```requirements.txt```, the ```code/``` folder, and Supplementary Tables S1-S5 excel documents.

## 0. Environment

Installs the pinned dependency set and `be_scan` into the runtime. In Colab,
also clones the repository if the notebook is running standalone.

In [ ]:
#@title Install dependencies { display-mode: "form" }
# Provisions the runtime from scratch on every run, so no persistent environment is required. Takes a few minutes on a cold Colab runtime.

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The repository holds the code, TableS1 and Inputs.
    # Set REPO_URL to clone from git, or mount Drive and point REPO_DIR at the copy there.
    REPO_DIR = "/content" # @param {type:"string"}
    REPO_DIR = Path(REPO_DIR)
    REPO_URL = "" # @param {type:"string"}

    if REPO_URL and not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    elif not REPO_DIR.exists():
        from google.colab import drive
        drive.mount("/content/drive")
        raise SystemExit(
            f"{REPO_DIR} not found. Set REPO_URL, or set REPO_DIR to the "
            "repository folder inside /content/drive."
        )
    os.chdir(REPO_DIR)

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "TableS1-ScreenData.xlsx").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# be_scan resolves its figure font from the working directory first.
if not (REPO_ROOT / "Arial.ttf").exists():
    subprocess.run([
        "wget", "-q",
        "https://raw.githubusercontent.com/liaulab/be-scan/main/be_scan/figure_plot/Arial.ttf",
        "-O", "Arial.ttf",
    ], check=False)

print("Environment ready.")

## 1. Imports

Every import used anywhere in the notebook.

In [ ]:
import sys
import warnings
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# be_scan figure primitives and their option dataclasses.
from be_scan.figure_plot import (
    AXIS, DOMAIN, LEGEND, LOLLIPOPSTYLE, OUTPUT, SCATTERSTYLE,
    DomainOpts,
    arial_font6,
    boxplot_figure,
    correlation_scatterplot_figure,
    jitterbox_kdeplot_figure,
    lollipop_figure,
    scatterplot_figure,
)
print("be_scan figure_plot loaded")

# Manuscript-specific code.
sys.path.insert(0, str(Path.cwd()))
# `code` is also a Python standard-library module.
sys.modules.pop("code", None)
from code import analysis as ana, boxplot as box, charts, config as cfg
from code import data as dat, dms as dms_mod, lollipop as lolli
from code import sankey as sankey_mod, structures as struct, trajectory as traj

# Option dataclasses for the figure types this manuscript adds, following the same convention as be_scan.
from code.figure_options import (
    BARSTYLE, BRACKET, HEATMAPSTYLE, HISTOGRAMSTYLE, PIESTYLE,
    SPLITLOLLIPOPSTYLE, TRAJECTORYSTYLE,
)
print("code/ modules loaded.")

# Chained-assignment and seaborn categorical warnings only.
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 2. Shared configuration

Genes, colors, domain boundaries, condition lists, cutoffs and output paths
are defined once in `code/config.py` and used by every section below.

In [ ]:
cfg.make_output_dirs()

GENES = cfg.GENES
COLOR_MAP = cfg.COLOR_MAP
DOMAINS = cfg.DOMAINS_LIST
XMAX = cfg.XMAX_DICT
MUT_PAL = cfg.MUT_PAL

print(f"{len(GENES)} genes: {', '.join(GENES)}")
print(f"Output root: {cfg.OUTPUTS_DIR}")
print(f"Screen data: {cfg.SCREEN_DATA_XLSX.name}")

In [ ]:
#@title Gene color palette { display-mode: "form" }
# The gene palette, saved so figure panels can be keyed to it by hand.

import seaborn as sns

sns.palplot(sns.color_palette(cfg.GENE_COLORS))
for i, gene in enumerate(GENES):
    plt.text(i, -0.2, gene, ha="center", va="top", rotation=0, fontsize=8)
plt.title("Gene color palette")
plt.show()
plt.savefig(cfg.OUTPUTS_DIR / "gene-palette.svg", dpi=1200, transparent=True)


## 3. Load screen data

All ten sheets of `TableS1-ScreenData.xlsx` is used throughout this notebook.

1. Drop the Excel index and blank spacer columns.
3. Add editor-agnostic `Editor` / `pos` / `muttype` / `mutations` columns, so downstream figures never branch on `AtoG_*` versus `CtoT_*`.

`pos` is the amino-acid position of the edit: `AtoG_pos` for ABE guides,
`CtoT_pos` for CBE guides. The same is used for `AtoG_muttype/CtoT_muttypes` and `AtoG_mutations/CtoT_mutations`.

In [ ]:
screens = dat.load_all_screens()

gof_abe, gof_cbe = screens[("gof", "ABE")], screens[("gof", "CBE")]
lof_abe, lof_cbe = screens[("lof", "ABE")], screens[("lof", "CBE")]
meki_abe, meki_cbe = screens[("meki", "ABE")], screens[("meki", "CBE")]
valid_abe, valid_cbe = screens[("valid", "ABE")], screens[("valid", "CBE")]
lof_erk_abe, lof_erk_cbe = screens[("lof_erk", "ABE")], screens[("lof_erk", "CBE")]

# Validation-ABE uses Day18
valid_abe = dat.rename_validation_abe(valid_abe)

# The LOF-ERK screen names its Z-score column per editor.
lof_erk_abe = lof_erk_abe.rename(columns={"Day14-ABE-Dox-UT-Z": cfg.LOF_ERK_COLUMN})
lof_erk_cbe = lof_erk_cbe.rename(columns={"Day14-CBE-Dox-UT-Z": cfg.LOF_ERK_COLUMN})

loaded = pd.DataFrame(
    [{"screen": s, "editor": e, "guides": len(df),
      "genes": df["Gene"].nunique(), "conditions": sum(c.endswith("-Z") for c in df.columns)}
     for (s, e), df in screens.items()]
)
loaded.to_csv(cfg.OUT_SUMMARIES / "screens-loaded.csv", index=False)
loaded

## 4. Per-condition scatterplots

Base-editing Z-score against amino-acid position, one panel per gene and
condition, ABE and CBE overlaid and colored by mutation type. Domains are
shaded behind the points.

In [ ]:
#@title Scatterplot style { display-mode: "form" }

SCATTER_STYLE = replace(
    SCATTERSTYLE,
    hue_col="muttype", color_map=cfg.MUT_PAL,
    marker_col="Editor", marker_map=cfg.EDITOR_MARKERS,
    transparency_threshold=3.0,
)
SCATTER_AXIS = replace(
    AXIS, xlabel="Amino Acid Position", ylabel="Base Editing Z-Score",
)
SCATTER_OUTPUT = replace(
    OUTPUT, figsize=cfg.FIGSIZE_FIVE_ACROSS,
    out_type="svg", dpi=1200, transparent=True, show=True, save=True,
)

SCATTER_LEGEND = replace(LEGEND, path=str(cfg.OUT_SCATTER / "mutation-type-legend"))
COMBINED_LEGEND = replace(LEGEND, path=str(cfg.OUT_SCATTER_COMBINED / "condition-legend"))


def scatter_panels(abe, cbe, ranges, conditions, out_dir, tag,
                   spacing=cfg.SCATTER_SPACING, style=SCATTER_STYLE,
                   figsize=cfg.FIGSIZE_FIVE_ACROSS, genes=None, suffix=""):
    """One panel per gene and condition. """
    rows = []

    for gene, gene_range, subset, present in ana.gene_panels(
            abe, cbe, ranges, conditions, genes=genes):
        domain_cfg = DomainOpts(gene_col="Gene", dom_setting=cfg.DOMAINS_LIST[gene])

        for condition in present:
            axis_cfg = replace(
                SCATTER_AXIS,
                title=f"{gene} {condition}",
                ylim=(gene_range[0], gene_range[1]),
                yticks=np.linspace(gene_range[0], gene_range[1], gene_range[2]).tolist(),
                xlim=(1 - spacing, XMAX[gene] + spacing),
                xticks=[1, XMAX[gene]],
            )
            output_cfg = replace(
                SCATTER_OUTPUT, figsize=figsize,
                path=str(out_dir / f"{tag}-{gene}-[{condition}]-Scatterplot{suffix}"),
            )
            scatterplot_figure(
                df_data=subset, y_col_list=[condition], x_col="pos",
                axis=axis_cfg, style=style, domain=domain_cfg, output=output_cfg,
                legend=SCATTER_LEGEND,
            )
            rows.append({"screen": tag, "Gene": gene, "condition": condition,
                         "n_guides": len(subset),
                         "min": subset[condition].min(), "max": subset[condition].max()})

    return pd.DataFrame(rows)

In [ ]:
#@title GOF screen { display-mode: "form" }

gof_scatter_stats = scatter_panels(
    gof_abe, gof_cbe,
    cfg.SCATTER_RANGES_GOF, cfg.GOF_SCREEN_COLUMNS,
    cfg.OUT_SCATTER, "GOF",
)
gof_scatter_stats.to_csv(cfg.OUT_SUMMARIES / "scatter-ranges-GOF.csv", index=False)
print(f"{len(gof_scatter_stats)} GOF panels")

In [ ]:
#@title GOF screen, enlarged MAPK1 panels { display-mode: "form" }

scatter_panels(
    gof_abe, gof_cbe,
    cfg.SCATTER_RANGES_GOF, cfg.GOF_SCREEN_COLUMNS,
    cfg.OUT_SCATTER, "GOF", genes=["MAPK1"],
    figsize=cfg.FIGSIZE_FIVE_ACROSS_LARGE, suffix="-Large",
);

In [ ]:
#@title MEKi screen { display-mode: "form" }

meki_scatter_stats = scatter_panels(
    meki_abe, meki_cbe,
    cfg.SCATTER_RANGES_MEKI, cfg.MEKI_SCREEN_COLUMNS,
    cfg.OUT_SCATTER, "MEKi",
)
meki_scatter_stats.to_csv(cfg.OUT_SUMMARIES / "scatter-ranges-MEKi.csv", index=False)
print(f"{len(meki_scatter_stats)} MEKi panels")

In [ ]:
#@title LOF hyperactivation screen { display-mode: "form" }

lof_scatter_stats = scatter_panels(
    lof_abe, lof_cbe,
    cfg.SCATTER_RANGES_LOF, cfg.LOF_SCREEN_COLUMNS,
    cfg.OUT_SCATTER, "LOF",
)
lof_scatter_stats.to_csv(cfg.OUT_SUMMARIES / "scatter-ranges-LOF.csv", index=False)
print(f"{len(lof_scatter_stats)} LOF panels")

In [ ]:
#@title LOF-ERK screen (MAPK1) { display-mode: "form" }
# The earlier LOF screen, which included ERK. MAPK1 only.

lof_erk_style = replace(SCATTER_STYLE, transparency_threshold=2.0)

lof_erk_scatter_stats = scatter_panels(
    lof_erk_abe, lof_erk_cbe,
    cfg.SCATTER_RANGES_LOF_ERK, [cfg.LOF_ERK_COLUMN],
    cfg.OUT_SCATTER, "LOF-ERK",
    style=lof_erk_style, figsize=cfg.FIGSIZE_FIVE_ACROSS_LARGE, genes=["MAPK1"],
)
lof_erk_scatter_stats

## 5. Combined-condition scatterplots

One panel per gene with every condition overlaid, colored by condition rather than by mutation type.

In [ ]:
#@title Combined-condition helper { display-mode: "form" }

COMBINED_STYLE_KWS = dict(
    marker_col="Editor", marker_map=cfg.EDITOR_MARKERS,
    transparency_threshold=3.0,
    transparent_kws={"edgecolor": "black", "alpha": 0.6, "s": 1.5, "linewidth": 0.1},
    opaque_kws={"edgecolor": "black", "alpha": 1, "s": 3, "linewidth": 0.25},
)

def combined_panels(frames, ranges, palette, out_dir, tag,
                    spacing=cfg.COMBINED_SPACING, title_suffix="All Conditions"):
    """One all-conditions panel per gene."""
    ranges = dict(zip(GENES, ranges))
    rows = []

    for gene in GENES:
        gene_range = ranges.get(gene)
        if not gene_range:
            continue

        plot_df = ana.combined_panel_frame(frames, gene)
        if plot_df is None:
            continue

        style_cfg = replace(SCATTERSTYLE, hue_col="Condition", color_map=palette,
                            **COMBINED_STYLE_KWS)
        axis_cfg = replace(
            SCATTER_AXIS, title=f"{gene} {title_suffix}",
            ylim=(gene_range[0], gene_range[1]),
            yticks=np.linspace(gene_range[0], gene_range[1], gene_range[2]).tolist(),
            xlim=(1 - spacing, XMAX[gene] + spacing), xticks=[1, XMAX[gene]],
        )
        output_cfg = replace(
            SCATTER_OUTPUT, figsize=cfg.FIGSIZE_FIVE_ACROSS,
            path=str(out_dir / f"{tag}-{gene}-[AllConditions]-Scatterplot"),
        )
        scatterplot_figure(
            df_data=plot_df, y_col_list=["score"], x_col="pos",
            axis=axis_cfg, style=style_cfg,
            domain=DomainOpts(gene_col="Gene", dom_setting=cfg.DOMAINS_LIST[gene]),
            output=output_cfg, legend=COMBINED_LEGEND,
        )
        rows.append({"screen": tag, "Gene": gene, "n_points": len(plot_df),
                     "min": plot_df["score"].min(), "max": plot_df["score"].max()})

    return pd.DataFrame(rows)

In [ ]:
#@title GOF, all conditions { display-mode: "form" }

combined_gof_stats = combined_panels(
    [(gof_abe, gof_cbe, cfg.COMBINED_GOF_COLUMNS)],
    cfg.COMBINED_RANGES_GOF, cfg.GOF_CONDITION_PAL,
    cfg.OUT_SCATTER_COMBINED, "GOF",
)
combined_gof_stats.to_csv(cfg.OUT_SUMMARIES / "combined-ranges-GOF.csv", index=False)
print(f"{len(combined_gof_stats)} panels")

In [ ]:
#@title MEKi, all conditions { display-mode: "form" }

combined_meki_stats = combined_panels(
    [(meki_abe, meki_cbe, cfg.COMBINED_MEKI_COLUMNS)],
    cfg.COMBINED_RANGES_MEKI, cfg.MEKI_CONDITION_PAL,
    cfg.OUT_SCATTER_COMBINED, "MEKi",
)
combined_meki_stats.to_csv(cfg.OUT_SUMMARIES / "combined-ranges-MEKi.csv", index=False)
print(f"{len(combined_meki_stats)} panels")

In [ ]:
#@title LOF, DMSO dropout beside hyperactivation { display-mode: "form" }

combined_lof_stats = combined_panels(
    [(gof_abe, gof_cbe, ["DMSO-Day0-Z"]),
     (lof_abe, lof_cbe, ["Dox-UT-Z"])],
    cfg.COMBINED_RANGES_LOF, cfg.LOF_CONDITION_PAL,
    cfg.OUT_SCATTER_COMBINED, "LOF", title_suffix="All LOF",
)
combined_lof_stats.to_csv(cfg.OUT_SUMMARIES / "combined-ranges-LOF.csv", index=False)
print(f"{len(combined_lof_stats)} panels")

## 6. GOF hits versus validation

A guide counts as validated when it passes the GOF cutoff (|Z| ≥ 3) and the validation cutoff (|Z| ≥ 2) in the same direction and condition.

In [ ]:
#@title Join validation onto GOF { display-mode: "form" }

gof_valid_abe = dat.merge_gof_validation(
    gof_abe, valid_abe, cfg.GOF_SCREEN_COLUMNS, cfg.VALID_CBE_SCREEN_COLUMNS)
gof_valid_cbe = dat.merge_gof_validation(
    gof_cbe, valid_cbe, cfg.GOF_SCREEN_COLUMNS, cfg.VALID_CBE_SCREEN_COLUMNS)

print(f"ABE {len(gof_valid_abe)} guides, CBE {len(gof_valid_cbe)} guides")

In [ ]:
#@title Hit recovery barplots { display-mode: "form" }

GOF_CONDITIONS = ana.GOF_CONDITIONS
VALID_CONDITIONS = ana.VALID_CONDITIONS

recovery = pd.concat([ana.hit_recovery(gof_valid_abe, "ABE"),
                      ana.hit_recovery(gof_valid_cbe, "CBE")], ignore_index=True)
recovery.to_csv(cfg.OUT_SUMMARIES / "gof-validation-hit-recovery.csv", index=False)
recovery

In [ ]:
#@title Barplots { display-mode: "form" }

BAR_COLORS = ["#90E0EF", "#FFCCCC", "#90E0EF", "#FFCCCC"]
BAR_LABELS = ["Pos Hits", "Pos Validation", "Neg Hits", "Neg Validation"]
BAR_LIMITS = {
    "ABE": dict(yticks_top=[0, 25, 50, 75, 100, 120], ylim_top=(0, 120),
                yticks_bot=[-80, -60, -40, -20, 0], ylim_bot=(-80, 0)),
    "CBE": dict(yticks_top=[0, 30, 60, 90], ylim_top=(0, 90),
                yticks_bot=[-80, -60, -40, -20, 0], ylim_bot=(-80, 0)),
}

FIG_OUTPUT = replace(OUTPUT, out_type="svg", dpi=1000, transparent=True,
                     show=True, save=True)

for editor in ("ABE", "CBE"):
    sub = recovery[recovery["editor"] == editor]
    charts.posneg_stacked_bar_chart(
        GOF_CONDITIONS,
        [(sub["pos_hits"] - sub["pos_validated"]).tolist(), sub["pos_validated"].tolist()],
        [(sub["neg_hits"] - sub["neg_validated"]).tolist(), sub["neg_validated"].tolist()],
        BAR_COLORS, BAR_LABELS, **BAR_LIMITS[editor],
        axis=replace(AXIS, xlabel="Conditions", ylabel="Hit Count",
                     title=f"{editor} GOF Hits and Validated Hits"),
        style=BARSTYLE,
        legend=replace(LEGEND, path=str(cfg.OUT_GOF_VS_VALID / f"{editor}-GOFvsValidation-barplot-legend")),
        output=replace(FIG_OUTPUT, figsize=(4 / 2.54, 6 / 2.54),
                       path=str(cfg.OUT_GOF_VS_VALID / f"{editor}-GOFvsValidation-barplot")),
    )

In [ ]:
#@title GOF versus validation correlation { display-mode: "form" }

CORR_AXES = {
    "ABE": dict(xlim=(-9, 18), ylim=(-12, 20), xticks=[-9, 0, 9, 18],
                yticks=[-12, -8, -4, 0, 4, 8, 12, 16, 20]),
    "CBE": dict(xlim=(-12, 24), ylim=(-14, 28), xticks=[-12, -6, 0, 6, 12, 18, 24],
                yticks=[-14, -7, 0, 7, 14, 21, 28]),
}

corr_style = replace(
    SCATTERSTYLE,
    transparent_kws={"edgecolor": "black", "alpha": 0.5, "s": 4, "linewidth": 0.1},
    opaque_kws={"edgecolor": "black", "alpha": 0.75, "s": 6, "linewidth": 0.1},
    line_kws={"color": "k", "ls": "--", "lw": 0.5},
    transparency_threshold=3,
)

correlations = []

for editor, frame in (("ABE", gof_valid_abe), ("CBE", gof_valid_cbe)):
    plot_df, stats = ana.correlation_frame(frame, editor)
    correlations.append(stats)

    correlation_scatterplot_figure(
        plot_df, "GOF Score", "Valid Score",
        palette=dict(zip(GOF_CONDITIONS, cfg.CORR_SCATTER_COLORS)),
        legend_order=GOF_CONDITIONS, hue_col="Condition",
        highlight_col="Gene", highlight_categories=[],
        axis=replace(AXIS, title=f"{editor} GOF vs Validation",
                     xlabel="GOF Score", ylabel="Validation Score",
                     **CORR_AXES[editor]),
        style=corr_style,
        output=replace(OUTPUT, figsize=(4 / 2.54, 3 / 2.54), save=True, rasterize=True,
                       path=str(cfg.OUT_GOF_VS_VALID / f"{editor}-GOFvsValidation-scatterplot")),
    )

correlations = pd.concat(correlations, ignore_index=True)
correlations.to_csv(cfg.OUT_SUMMARIES / "gof-validation-correlations.csv", index=False)
correlations

## 7. Pocket hits — co-crystal structures

For each inhibited node, the minimum heavy-atom distance from the bound
inhibitor to every residue is measured, and resistance hits are classified as inside the drug-binding pocket (a residue within 4 Å of the ligand lies within one residue of the edit), outside it, or  unresolved in the given structure.

In [ ]:
#@title Pocket analysis helpers { display-mode: "form" }

pocket_analysis = ana.pocket_analysis
pocket_summary = ana.pocket_summary
stacked_series = ana.stacked_series
structure_labels = ana.structure_labels
hit_distance_frame = ana.hit_distance_frame
hit_score_frame = ana.hit_score_frame

In [ ]:
#@title Measure ligand distances (co-crystals) { display-mode: "form" }

pdb_distances = struct.load_pocket_distances(cfg.PDB_STRUCTURES, cfg.OUT_POCKET_PDB)
pdb_records = pocket_analysis(cfg.PDB_STRUCTURES, cfg.PDB_PARALOGS,
                              pdb_distances, gof_abe, gof_cbe)

pdb_summary = pocket_summary(pdb_records)
pdb_summary.to_csv(cfg.OUT_SUMMARIES / "pocket-hits-PDB.csv", index=False)
pdb_summary

In [ ]:
#@title Stacked bars (co-crystals) { display-mode: "form" }

POCKET_KEYS = ["InPocket", "OutPocket", "Unresolved"]
POCKET_COLORS = [cfg.POCKET_PDB_COLORS[k] for k in POCKET_KEYS]
POCKET_LABELS = ["In Pocket", "Outside Pocket", "Unresolved"]
OUTSIDE = cfg.POCKET_EXTRA_COLORS["OutsideNode"]

BAR_TITLE = "Number of Hit Guides, 4.0 A"


def pocket_bars(records, out_dir, tag, keys, colors, labels, bar_width):
    """The four bar variants each pocket analysis produces."""
    xlabels = ana.structure_labels(records)
    series = ana.stacked_series(records, keys)
    outside = ana.outside_node_series(records)

    charts.stacked_bar_chart(
        xlabels, series, colors, labels,
        axis=replace(AXIS, ylabel="Hit Count", title=BAR_TITLE,
                     ylim=(0, 40), yticks=[0, 10, 20, 30, 40]),
        style=BARSTYLE,
        legend=replace(LEGEND, path=str(out_dir / f"{tag}-StackedBar-legend")),
        output=replace(FIG_OUTPUT, figsize=(bar_width, 4 / 2.54),
                       path=str(out_dir / f"{tag}-StackedBar")),
    )

    charts.stacked_bar_chart(
        xlabels, series + [outside], colors + [OUTSIDE], labels + ["Outside Node"],
        axis=replace(AXIS, ylabel="Hit Count", title=BAR_TITLE,
                     ylim=(0, 120), yticks=[0, 20, 40, 60, 80, 100, 120]),
        style=BARSTYLE,
        legend=replace(LEGEND, path=str(out_dir / f"{tag}-ExtraGenes-StackedBar-legend")),
        output=replace(FIG_OUTPUT, figsize=(bar_width, 5 / 2.54),
                       path=str(out_dir / f"{tag}-ExtraGenes-StackedBar")),
    )

    charts.stacked_bar_chart(
        xlabels, series + [outside], colors + [OUTSIDE], labels + ["Outside Node"],
        axis=replace(AXIS, ylabel="Hit Count", title=BAR_TITLE,
                     ylim=(0, 120), yticks=[0, 10, 100]),
        style=replace(BARSTYLE, ylog=True),
        legend=replace(LEGEND, path=str(out_dir / f"{tag}-ExtraGenes-log-StackedBar-legend")),
        output=replace(FIG_OUTPUT, figsize=(bar_width, 4 / 2.54),
                       path=str(out_dir / f"{tag}-ExtraGenes-log-StackedBar")),
    )

    charts.broken_stacked_bar_chart(
        xlabels, series + [outside], colors + [OUTSIDE], labels + ["Outside Node"],
        axis=replace(AXIS, ylabel="Hit Count", title=BAR_TITLE,
                     ylim=(0, 120), yticks=[0, 20, 40, 60, 80, 100, 120]),
        style=replace(BARSTYLE, ybreak=40, ybreak_gap=5, ybreak_ratio=(1, 1)),
        legend=replace(LEGEND, path=str(out_dir / f"{tag}-ExtraGenes-BrokenBar-legend")),
        output=replace(FIG_OUTPUT, figsize=(bar_width + 0.2, 3.6 / 2.54),
                       path=str(out_dir / f"{tag}-ExtraGenes-BrokenBar")),
    )


pocket_bars(pdb_records, cfg.OUT_POCKET_PDB, "HitGuides-5Nodes",
            POCKET_KEYS, POCKET_COLORS, POCKET_LABELS, 4 / 2.54)

In [ ]:
#@title Distance histograms and boxplots (co-crystals) { display-mode: "form" }

PDB_HIST_TICKS = {
    "PTPN11 7JVM": ([0, 10, 20, 30, 40], [0, 5, 10]),
    "KRAS 6UT0": ([0, 10, 20, 30], [0, 5, 10, 15]),
    "BRAF 5C9C": ([0, 10, 20], [0, 1, 2]),
    "MAP2K1 7JUR": ([0, 10, 20, 30], [0, 5, 10, 15]),
    "MAPK1 6RQ4": ([0, 10, 20, 30], [0, 5, 10, 15]),
}
PDB_BOX_TICKS = {
    "PTPN11 7JVM": [0, 5, 10, 15, 20],
    "KRAS 6UT0": [0, 5, 10, 15, 20, 25],
    "BRAF 5C9C": [2, 3, 4, 5],
    "MAP2K1 7JUR": [0, 5, 10, 15, 20, 25],
    "MAPK1 6RQ4": [2, 4, 6, 8, 10, 12],
}

for record in pdb_records:
    name = record["label"].replace(" ", "-")
    xticks, yticks = PDB_HIST_TICKS[record["label"]]

    charts.distance_histogram_figure(
        ana.hit_distance_frame(record), "distance", "editor", cfg.EDITOR_PALETTE,
        axis=replace(AXIS, xlabel="Distance", ylabel="Hits Count", title=record["label"],
                     xlim=(xticks[0], xticks[-1]), xticks=xticks,
                     ylim=(yticks[0], yticks[-1]), yticks=yticks),
        style=HISTOGRAMSTYLE,
        legend=replace(LEGEND, path=str(cfg.OUT_POCKET_PDB / f"{name}-DistanceHistogram-legend")),
        output=replace(FIG_OUTPUT, figsize=(8 / 2.54, 2 / 2.54),
                       path=str(cfg.OUT_POCKET_PDB / f"{name}-DistanceHistogram")),
    )

    box_ticks = PDB_BOX_TICKS[record["label"]]
    scores = ana.hit_score_frame(record, POCKET_KEYS)
    boxplot_figure(
        scores, "pocket", "Z-Score", cfg.POCKET_PDB_COLORS, lab_order=POCKET_KEYS,
        axis=replace(AXIS, xlabel="Pocket class", ylabel="Z-Score",
                     title=f"{name} Hits In/Out Pocket",
                     ylim=(box_ticks[0], box_ticks[-1]), yticks=box_ticks),
        output=replace(FIG_OUTPUT, figsize=(2.6 / 2.54, 4 / 2.54),
                       path=str(cfg.OUT_POCKET_PDB / f"{name}-GroupedBoxplot")),
    )
    charts.patch_legend(
        cfg.POCKET_PDB_COLORS,
        replace(LEGEND, path=str(cfg.OUT_POCKET_PDB / f"{name}-GroupedBoxplot-legend")),
        FIG_OUTPUT)

In [ ]:
#@title Pie charts (co-crystals) { display-mode: "form" }

ALL_PIE_LABELS = POCKET_LABELS + ["In Paralogs", "Other Genes"]
ALL_PIE_COLORS = POCKET_COLORS + [cfg.POCKET_EXTRA_COLORS["InParalogs"],
                                  cfg.POCKET_EXTRA_COLORS["OtherGenes"]]

for record in pdb_records:
    name = record["label"].replace(" ", "-")
    for editor in ("ABE", "CBE"):
        charts.pie_chart_figure(
            [record[f"n_{editor}_{k}"] for k in POCKET_KEYS],
            POCKET_LABELS, POCKET_COLORS,
            axis=replace(AXIS, title=f"{name}-{editor}"), style=PIESTYLE,
            legend=replace(LEGEND, path=str(cfg.OUT_POCKET_PDB / f"{name}-{editor}-PieChart-legend")),
            output=replace(FIG_OUTPUT, figsize=(3 / 2.54, 3 / 2.54),
                           path=str(cfg.OUT_POCKET_PDB / f"{name}-{editor}-PieChart")),
        )
        charts.pie_chart_figure(
            [record[f"n_{editor}_{k}"] for k in POCKET_KEYS]
            + [record["off_node"]["paralog"], record["off_node"]["other"]],
            ALL_PIE_LABELS, ALL_PIE_COLORS,
            axis=replace(AXIS, title=f"{name}-{editor}"), style=PIESTYLE,
            legend=replace(LEGEND, path=str(cfg.OUT_POCKET_PDB / f"{name}-{editor}-PieChart-All-legend")),
            output=replace(FIG_OUTPUT, figsize=(3 / 2.54, 3 / 2.54),
                           path=str(cfg.OUT_POCKET_PDB / f"{name}-{editor}-PieChart-All")),
        )

## 8. Pocket hits — AlphaFold models

The same analysis extended to every paralog, using AlphaFold models with the inhibitor aligned into the pocket. This covers nodes with no co-crystal structure (NRAS, HRAS, MRAS, ARAF, RAF1, MAP2K2, MAPK3) and provides full length models for nodes with a co-crystal structure (SHP2, KRAS, BRAF, MAP2K1, MAPK1).

In [ ]:
#@title Measure ligand distances (AlphaFold) { display-mode: "form" }

af_distances = struct.load_pocket_distances(cfg.AF_STRUCTURES, cfg.OUT_POCKET_AF)
af_records = pocket_analysis(cfg.AF_STRUCTURES, cfg.AF_PARALOGS,
                             af_distances, gof_abe, gof_cbe)

af_summary = pocket_summary(af_records)
af_summary.to_csv(cfg.OUT_SUMMARIES / "pocket-hits-AlphaFold.csv", index=False)
af_summary

In [ ]:
#@title Stacked bars (AlphaFold) { display-mode: "form" }

AF_KEYS = ["InPocket", "OutPocket"]
AF_COLORS = [cfg.POCKET_AF_COLORS[k] for k in AF_KEYS]
AF_LABELS = ["In Pocket", "Out Pocket"]

pocket_bars(af_records, cfg.OUT_POCKET_AF, "HitGuides-AF",
            POCKET_KEYS, POCKET_COLORS, POCKET_LABELS, 12 / 2.54)

In [ ]:
#@title Distance histograms and boxplots (AlphaFold) { display-mode: "form" }

AF_HIST_TICKS = {
    "PTPN11": ([0, 10, 20, 30], [0, 5, 10]),
    "KRAS": ([0, 10, 20], [0, 5, 10]),
    "NRAS": ([0, 25, 50], [0, 1, 2]),
    "HRAS": ([0, 20, 40], [0, 1, 2, 3]),
    "MRAS": ([0, 5, 10, 15, 20], [0, 1, 2, 3]),
    "ARAF": ([0, 20, 40, 60], [0, 1, 2, 3]),
    "BRAF": ([0, 20, 40, 60, 80], [0, 1, 2]),
    "RAF1": ([0, 25, 50], [0, 1, 2]),
    "MAP2K1": ([0, 10, 20, 30], [0, 5, 10]),
    "MAP2K2": ([0, 20, 40], [0, 2, 4, 6, 8]),
    "MAPK1": ([0, 10, 20, 30], [0, 2, 4, 6, 8]),
    "MAPK3": ([0, 10, 20, 30], [0, 1, 2, 3, 4, 5]),
}
AF_BOX_TICKS = {
    "PTPN11": [0, 5, 10, 15, 20], "KRAS": [0, 5, 10, 15, 20, 25],
    "NRAS": [0, 5, 10, 15], "HRAS": [2, 4, 6], "MRAS": [2, 4, 6],
    "ARAF": [0, 5, 10, 15, 20], "BRAF": [2, 4, 6], "RAF1": [2, 4, 6],
    "MAP2K1": [0, 5, 10, 15, 20], "MAP2K2": [2, 4, 6, 8, 10],
    "MAPK1": [2, 4, 6, 8, 10, 12], "MAPK3": [2, 4, 6, 8, 10],
}

for record in af_records:
    name = record["label"].replace(" ", "-")
    xticks, yticks = AF_HIST_TICKS[record["gene"]]

    charts.distance_histogram_figure(
        ana.hit_distance_frame(record), "distance", "editor", cfg.EDITOR_PALETTE,
        axis=replace(AXIS, xlabel="Distance", ylabel="Hits Count", title=record["label"],
                     xlim=(xticks[0], xticks[-1]), xticks=xticks,
                     ylim=(yticks[0], yticks[-1]), yticks=yticks),
        style=HISTOGRAMSTYLE,
        legend=replace(LEGEND, path=str(cfg.OUT_POCKET_AF / f"{name}-AF-DistanceHistogram-legend")),
        output=replace(FIG_OUTPUT, figsize=(8 / 2.54, 2 / 2.54),
                       path=str(cfg.OUT_POCKET_AF / f"{name}-AF-DistanceHistogram")),
    )

    box_ticks = AF_BOX_TICKS[record["gene"]]
    boxplot_figure(
        ana.hit_score_frame(record, AF_KEYS), "pocket", "Z-Score",
        cfg.POCKET_AF_COLORS, lab_order=AF_KEYS,
        axis=replace(AXIS, xlabel="Pocket class", ylabel="Z-Score",
                     title=f"{name} Hits In/Out Pocket",
                     ylim=(box_ticks[0], box_ticks[-1]), yticks=box_ticks),
        output=replace(FIG_OUTPUT, figsize=(2.6 / 2.54, 4 / 2.54),
                       path=str(cfg.OUT_POCKET_AF / f"{name}-AF-GroupedBoxplot")),
    )
    charts.patch_legend(
        cfg.POCKET_AF_COLORS,
        replace(LEGEND, path=str(cfg.OUT_POCKET_AF / f"{name}-AF-GroupedBoxplot-legend")),
        FIG_OUTPUT)

In [ ]:
#@title Pie charts (AlphaFold) { display-mode: "form" }

AF_ALL_LABELS = AF_LABELS + ["In Paralogs", "Other Genes"]
AF_ALL_COLORS = AF_COLORS + [cfg.POCKET_EXTRA_COLORS["InParalogs"],
                             cfg.POCKET_EXTRA_COLORS["OtherGenes"]]

for record in af_records:
    name = record["label"].replace(" ", "-")
    for editor in ("ABE", "CBE"):
        charts.pie_chart_figure(
            [record[f"n_{editor}_{k}"] for k in AF_KEYS], AF_LABELS, AF_COLORS,
            axis=replace(AXIS, title=f"{name}-{editor}"), style=PIESTYLE,
            legend=replace(LEGEND, path=str(cfg.OUT_POCKET_AF / f"{name}-AF-{editor}-PieChart-legend")),
            output=replace(FIG_OUTPUT, figsize=(3 / 2.54, 3 / 2.54),
                           path=str(cfg.OUT_POCKET_AF / f"{name}-AF-{editor}-PieChart")),
        )
        charts.pie_chart_figure(
            [record[f"n_{editor}_{k}"] for k in AF_KEYS]
            + [record["off_node"]["paralog"], record["off_node"]["other"]],
            AF_ALL_LABELS, AF_ALL_COLORS,
            axis=replace(AXIS, title=f"{name}-{editor}"), style=PIESTYLE,
            legend=replace(LEGEND, path=str(cfg.OUT_POCKET_AF / f"{name}-AF-{editor}-PieChart-All-legend")),
            output=replace(FIG_OUTPUT, figsize=(3 / 2.54, 3 / 2.54),
                           path=str(cfg.OUT_POCKET_AF / f"{name}-AF-{editor}-PieChart-All")),
        )

In [ ]:
#@title Pie charts pooled by paralog family { display-mode: "form" }
# Every RAS isoform shares one inhibitor, so pooling the family gives the
# per-inhibitor picture the per-gene pies split apart.

FAMILY_LABELS = AF_LABELS + ["Other Genes"]
FAMILY_COLORS = AF_COLORS + [cfg.POCKET_EXTRA_COLORS["OtherGenes"]]
by_gene = {r["gene"]: r for r in af_records}

for family, members in ana.PARALOG_FAMILIES.items():
    for editor in ("ABE", "CBE"):
        charts.pie_chart_figure(
            [sum(by_gene[g][f"n_{editor}_InPocket"] for g in members),
             sum(by_gene[g][f"n_{editor}_OutPocket"] for g in members),
             by_gene[members[0]]["off_node"]["other"]],
            FAMILY_LABELS, FAMILY_COLORS,
            axis=replace(AXIS, title=f"{family}-{editor}"), style=PIESTYLE,
            legend=replace(LEGEND, path=str(cfg.OUT_POCKET_AF / f"{family}-AF-{editor}-PieChart-All-legend")),
            output=replace(FIG_OUTPUT, figsize=(3 / 2.54, 3 / 2.54),
                           path=str(cfg.OUT_POCKET_AF / f"{family}-AF-{editor}-PieChart-All")),
        )

## 9. Splice-guide validation heatmaps

Mean Z-score per gene and condition for the essential-splice-site guides in the validation library. Each panel is drawn twice: across all genes, and across the three RAF paralogs alone.

In [ ]:
#@title Splice validation heatmaps { display-mode: "form" }

splice_abe = dat.to_protein_names(dat.splice_guides(valid_abe))
splice_cbe = dat.to_protein_names(dat.splice_guides(valid_cbe))
PROTEINS = cfg.PROTEINS

DMSO_NORM = cfg.VALID_CBE_SCREEN_COLUMNS[:6]
DAY0_NORM = cfg.VALID_CBE_SCREEN_COLUMNS[:1] + cfg.VALID_CBE_SCREEN_COLUMNS[6:]
LABELS = cfg.HEATMAP_CONDITION_LABELS

# Every heatmap is drawn twice: across all genes and across the three RAF paralogs alone.
PANEL_SETS = [
    ("", PROTEINS, cfg.SPLICE_HEATMAP_FIGSIZE),
    ("-RAF", cfg.SPLICE_HEATMAP_RAF_PROTEINS, cfg.SPLICE_HEATMAP_RAF_FIGSIZE),
]


def splice_heatmap(frame, columns, name):
    """Draw the pathway-wide panel and its RAF-only counterpart."""
    renamed = frame.rename(columns=dict(zip(columns, LABELS)))
    pivots = {}
    for suffix, genes, figsize in PANEL_SETS:
        stem = f"{name}{suffix}"
        _, _, pivot = charts.gene_condition_heatmap(
            renamed, LABELS[::-1], genes, gene_col="Gene",
            axis=replace(AXIS, xlabel="Gene", ylabel="Condition"),
            style=HEATMAPSTYLE,
            legend=replace(LEGEND, path=str(cfg.OUT_SPLICE_HEATMAP / f"{stem}-legend")),
            output=replace(FIG_OUTPUT, figsize=figsize, dpi=1200,
                           path=str(cfg.OUT_SPLICE_HEATMAP / stem)),
        )
        pivots[suffix] = pivot
    return pivots


panels = [
    (splice_abe, DMSO_NORM, "SpliceValidation-ABE-byGene-heatmap"),
    (splice_cbe, DMSO_NORM, "SpliceValidation-CBE-byGene-heatmap"),
    (splice_abe, DAY0_NORM, "SpliceValidation-ABE-byGene-Day0Norm-heatmap"),
    (splice_cbe, DAY0_NORM, "SpliceValidation-CBE-byGene-Day0Norm-heatmap"),
]

for frame, columns, name in panels:
    splice_heatmap(frame, columns, name)

pooled = pd.concat([
    splice_abe[["sgRNA_ID", "sgRNA_seq", "Gene"] + DAY0_NORM],
    splice_cbe[["sgRNA_ID", "sgRNA_seq", "Gene"] + DAY0_NORM],
], ignore_index=True)

pooled_pivots = splice_heatmap(
    pooled, DAY0_NORM, "SpliceValidation-byGene-Day0Norm-heatmap")

pooled_pivots[""].to_csv(cfg.OUT_SUMMARIES / "splice-validation-means.csv")
pooled_pivots["-RAF"].to_csv(cfg.OUT_SUMMARIES / "splice-validation-means-RAF.csv")
pooled_pivots["-RAF"].round(2)

## 10. Sankey — validated hits by inhibitor

Where each inhibitor's resistance hits fall in the pathway. Links are colored by position relative to the inhibited node: gray on-target, pink downstream, blue upstream.

In [ ]:
#@title Build hit connections and draw the Sankey { display-mode: "form" }

connections = sankey_mod.build_hit_connections(
    gof_valid_abe, gof_valid_cbe,
    cfg.GOF_DMSO_COLUMNS, [f"Valid-{c}" for c in cfg.GOF_DMSO_COLUMNS],
)
connections.to_csv(cfg.OUT_SUMMARIES / "sankey-hit-connections.csv", index=False)

direction = sankey_mod.directional_summary(
    connections[connections["hit_count"] >= cfg.SANKEY_MIN_HIT_COUNT])
direction.to_csv(cfg.OUT_SUMMARIES / "sankey-hit-directions.csv", index=False)

figure = sankey_mod.sankey_figure(connections, min_hit_count=cfg.SANKEY_MIN_HIT_COUNT)
sankey_mod.save_sankey(figure, cfg.OUT_SANKEY / "MAPK-GOF-Validated-Sankey")
figure.show()

direction

## 11. GMM trajectory

Guides are clustered on their inhibitor-response profiles with a Gaussian mixture model. Component means are joined by a minimum spanning tree, and distance from a root component along that tree gives each guide a pseudotime. PCA is used only to display the result and not used for fitting.

In [ ]:
#@title Validation screen trajectory { display-mode: "form" }

CONDITIONS = cfg.TRAJECTORY_VALID_CONDITIONS

# Day-0 filtering, control removal, GOF annotation and the reproducible row
# ordering all live in code/analysis.py.
valid_input = ana.validation_trajectory_input(valid_abe, valid_cbe, gof_abe, gof_cbe)

print(f"{len(valid_input)} guides enter the trajectory")

valid_fit = traj.fit_trajectory(
    valid_input[CONDITIONS],
    n_components=cfg.TRAJECTORY_VALID_N_COMPONENTS,
    trajectory_root=cfg.TRAJECTORY_VALID_ROOT,
)
print(f"converged={valid_fit['converged']}  BIC={valid_fit['bic']:.1f}  "
      f"AIC={valid_fit['aic']:.1f}  "
      f"PC1={valid_fit['explained_variance'][0]*100:.1f}%  "
      f"PC2={valid_fit['explained_variance'][1]*100:.1f}%")

valid_traj_summary = traj.trajectory_summary(valid_fit)
valid_traj_summary.to_csv(cfg.OUT_SUMMARIES / "trajectory-validation-clusters.csv", index=False)
valid_traj_summary

FIG_OUTPUT = replace(OUTPUT, out_type="svg", dpi=1000, transparent=True,
                     show=True, save=True)

In [ ]:
#@title Validation trajectory figures { display-mode: "form" }

valid_table = traj.plot_trajectory(
    valid_fit, valid_input["Gene"],
    trajectory_clusters=cfg.TRAJECTORY_VALID_SUBSETS,
    annotations=valid_input[["mutations", "DMSO-Day0-Z"]],
    style=replace(TRAJECTORYSTYLE, gray_gene_bars=True),
    output=replace(FIG_OUTPUT, dpi=1000,
                   path=str(cfg.OUT_TRAJECTORY / "Validation" / "GMM_Trajectory-Validation")),
)

valid_composition = traj.cluster_gene_composition(valid_fit, valid_input["Gene"])
valid_composition.to_csv(cfg.OUT_SUMMARIES / "trajectory-validation-composition.csv", index=False)
valid_composition.head(20)

In [ ]:
#@title MEKi screen trajectory { display-mode: "form" }

MEKI_CONDITIONS = cfg.TRAJECTORY_MEKI_CONDITIONS

meki_input = ana.meki_trajectory_input(meki_abe, meki_cbe, gof_abe, gof_cbe)

print(f"{len(meki_input)} guides enter the trajectory")

meki_fit = traj.fit_trajectory(
    meki_input[cfg.TRAJECTORY_MEKI_LABELS],
    n_components=cfg.TRAJECTORY_MEKI_N_COMPONENTS,
    trajectory_root=cfg.TRAJECTORY_MEKI_ROOT,
)
print(f"converged={meki_fit['converged']}  BIC={meki_fit['bic']:.1f}  "
      f"PC1={meki_fit['explained_variance'][0]*100:.1f}%  "
      f"PC2={meki_fit['explained_variance'][1]*100:.1f}%")

meki_traj_summary = traj.trajectory_summary(meki_fit)
meki_traj_summary.to_csv(cfg.OUT_SUMMARIES / "trajectory-meki-clusters.csv", index=False)
meki_traj_summary

In [ ]:
#@title MEKi trajectory figures { display-mode: "form" }

meki_table = traj.plot_trajectory(
    meki_fit, meki_input["Gene"],
    annotations=meki_input[["mutations"]],
    per_cluster_panels=False,
    style=replace(TRAJECTORYSTYLE, heatmap_figsize=(12 / 2.54, 6 / 2.54)),
    output=replace(FIG_OUTPUT, dpi=1000,
                   path=str(cfg.OUT_TRAJECTORY / "MEKi" / "GMM_Trajectory-MEKi")),
)

meki_composition = traj.cluster_gene_composition(meki_fit, meki_input["Gene"])
meki_composition.to_csv(cfg.OUT_SUMMARIES / "trajectory-meki-composition.csv", index=False)
meki_composition.head(20)

## 12. LOF hyperactivation versus DMS

Guides that hyperactivate the pathway when the target is disrupted, compared against published deep mutational scanning data for the same protein. A guide making several edits takes the minimum DMS score across them, since all three DMS datasets score depletion negatively.

DMS sources: PTPN11 (Jiang, *Nat Commun* 2025), KRAS (integrated dataset, HCC827 G12D), MAPK1/ERK2 (Brenan, *Cell Rep* 2016).

In [ ]:
#@title Load DMS data and score guides { display-mode: "form" }

dms_dicts = dms_mod.load_all_dms()
dms_stats = dms_mod.dms_summary_stats(dms_dicts)
dms_stats.to_csv(cfg.OUT_SUMMARIES / "dms-dataset-stats.csv")

lof_gof_long = dat.build_lof_gof_long(gof_abe, gof_cbe, lof_abe, lof_cbe)
dms_frame = dms_mod.prepare_dms_frame(lof_gof_long, dms_dicts, dms_stats)
dms_frame.to_csv(cfg.OUT_DMS / "LOF-DMS-merged.csv", index=False)

dms_stats

In [ ]:
#@title LOF-ERK frame, scored against the ERK2 DMS { display-mode: "form" }

lof_erk_long = pd.concat([lof_erk_abe, lof_erk_cbe], ignore_index=True)
lof_erk_long = lof_erk_long[lof_erk_long["mutations"].notna()]
lof_erk_dms = dms_mod.add_dms_scores(lof_erk_long, dms_dicts, dms_stats)
lof_erk_dms.to_csv(cfg.OUT_DMS / "LOF-ERK-DMS-merged.csv", index=False)

print(f"{len(lof_erk_dms)} LOF-ERK guides, "
      f"{lof_erk_dms['DMS-Score-Z'].notna().sum()} with a DMS score")

In [ ]:
#@title Lollipops: hyperactivation above DMS score below { display-mode: "form" }

LOLLI_LIMITS = {"PTPN11": ((-6, 6), [-6, 0, 6]), "KRAS": ((-4, 12), [-4, 0, 4, 8, 12])}
LOLLI_COLORS_POS = ["#ff5a5a", "#feb1c6"]
LOLLI_COLORS_NEG = ["#afffb9", "#5aa9e6"]


def dms_lollipop(frame, gene, score_col, ylim, yticks, out_name, pos_threshold=2):
    positioned = dat.drop_unpositioned(frame[frame["Gene"] == gene])
    if positioned.empty:
        print(f"  {gene}: no positioned guides, skipped")
        return
    top = positioned[["pos", "Editor", "Gene", score_col]].dropna(subset=[score_col])
    bottom = positioned.loc[positioned[score_col] > pos_threshold,
                            ["pos", "Editor", "Gene", "DMS-Score-Z"]].dropna(
                                subset=["DMS-Score-Z"])

    lollipop_figure(
        top, "pos", score_col, pos_threshold, "Editor", LOLLI_COLORS_POS,
        bottom, "pos", "DMS-Score-Z", -1, "Editor", LOLLI_COLORS_NEG,
        axis=replace(AXIS, title=f"{gene} - LOF hyperactivation vs DMS",
                     ylim=ylim, yticks=yticks,
                     xlim=(1 - 10, XMAX[gene] + 10), xticks=[1, XMAX[gene]],
                     ylabel="Z-Score", xlabel="Amino Acid Position", linewidth=0.5),
        domain=replace(DOMAIN, default_alpha=0.25, dom_setting=cfg.DOMAINS_LIST[gene]),
        style=replace(LOLLIPOPSTYLE, marker_rasterized=True),
        output=replace(OUTPUT, figsize=(2.5, 0.75), save=True,
                       path=str(cfg.OUT_DMS / out_name)),
    )


for gene, (ylim, yticks) in LOLLI_LIMITS.items():
    dms_lollipop(dms_frame, gene, "Dox-UT-Z", ylim, yticks,
                 f"{gene}-LOF-Hyperactivation-DMS-Lollipop")

dms_lollipop(lof_erk_dms, "MAPK1", cfg.LOF_ERK_COLUMN, (-6, 6), [-6, 0, 6],
             "MAPK1-LOF-ERK-Hyperactivation-DMS-Lollipop")

In [ ]:
#@title Quartile boxplots { display-mode: "form" }

np.random.seed(cfg.RANDOM_STATE)

def quartile_boxplot(frame, value_col, quartile_col, ylim, yticks, title, out_name):
    """Distribution of one score across quartiles of the other."""
    usable = frame.dropna(subset=[value_col, quartile_col]).copy()
    if usable[quartile_col].nunique() < 4:
        print(f"  {title}: too few distinct values to form quartiles, skipped")
        return
    usable["quartile"] = pd.qcut(usable[quartile_col], q=4, labels=False) + 1

    jitterbox_kdeplot_figure(
        usable.reset_index(drop=True), xcol="quartile", ycol=value_col,
        huecol="Gene", color_dict=cfg.COLOR_MAP,
        axis=replace(AXIS, title=title, ylim=ylim, yticks=yticks,
                     ylabel=value_col, xlabel=f"{quartile_col} quartile"),
        output=replace(OUTPUT, figsize=(3.5 / 2.54, 3.5 / 2.54), save=True,
                       path=str(cfg.OUT_DMS / out_name)),
    )


HYPER_LIMITS = ((-8, 12), np.linspace(-8, 12, 6).tolist())
DMS_LIMITS = ((-6, 6), np.linspace(-6, 6, 5).tolist())

for gene in ("PTPN11", "KRAS"):
    subset = dms_frame[dms_frame["Gene"] == gene]
    quartile_boxplot(subset, "Dox-UT-Z", "DMS-Score-Z", *HYPER_LIMITS,
                     f"MAPK - {gene}", f"{gene}-Hyperactivation-by-DMS-quartile")
    quartile_boxplot(subset, "DMS-Score-Z", "Dox-UT-Z", *DMS_LIMITS,
                     f"MAPK - {gene}", f"{gene}-DMS-by-Hyperactivation-quartile")

quartile_boxplot(dms_frame, "Dox-UT-Z", "DMS-Score-Z", *HYPER_LIMITS,
                 "MAPK", "All-Hyperactivation-by-DMS-quartile")
quartile_boxplot(dms_frame, "DMS-Score-Z", "Dox-UT-Z", *DMS_LIMITS,
                 "MAPK", "All-DMS-by-Hyperactivation-quartile")

quartile_boxplot(lof_erk_dms[lof_erk_dms["Gene"] == "MAPK1"], cfg.LOF_ERK_COLUMN,
                 "DMS-Score-Z", (-4, 4), np.linspace(-4, 4, 5).tolist(),
                 "MAPK - MAPK1", "MAPK1-LOF-ERK-Hyperactivation-by-DMS-quartile")
quartile_boxplot(lof_erk_dms[lof_erk_dms["Gene"] == "MAPK1"], "DMS-Score-Z",
                 cfg.LOF_ERK_COLUMN, (-4, 4), np.linspace(-4, 4, 5).tolist(),
                 "MAPK - MAPK1", "MAPK1-LOF-ERK-DMS-by-Hyperactivation-quartile")

In [ ]:
#@title Hyperactivation versus DMS correlation { display-mode: "form" }

# Editor is carried by marker shape; every point takes the same colour.
#
# Colouring by editor reused MUT_PAL's exact hexes -- #FDB462 is Missense
# there and the old CBE #80B1D3 is Silent -- so these panels read as though
# they showed mutation classes. They do not: every point here is missense,
# because only missense variants exist in the DMS datasets to match against.
#
# be_scan's correlation figure groups solely by `hue_col` and never reads
# style.marker_col / marker_map, so the shape has to reach seaborn through
# the kwargs it splats into scatterplot(). marker_col/marker_map are kept
# below because they state the intent and other figures do honour them.
DMS_POINT_COLOR = "#FDB462"
EDITOR_SHAPE_KWS = {"style": "Editor", "markers": dict(cfg.EDITOR_MARKERS)}

dms_corr_style = replace(
    SCATTERSTYLE,
    transparent_kws={"edgecolor": "black", "alpha": 0.25, "s": 4,
                     "linewidth": 0.1, **EDITOR_SHAPE_KWS},
    opaque_kws={"edgecolor": "black", "alpha": 0.75, "s": 6,
                "linewidth": 0.1, **EDITOR_SHAPE_KWS},
    line_kws={"color": "k", "ls": "--", "lw": 0.5},
    transparency_threshold=0,
    marker_col="Editor", marker_map=cfg.EDITOR_MARKERS,
)
EDITOR_COLORS = {"ABE": DMS_POINT_COLOR, "CBE": DMS_POINT_COLOR}


def write_editor_legend(path):
    """Replace be_scan's colour legend with one keyed by marker shape.

    be_scan builds its legend entries as circles differing only in face
    colour; with one colour for both editors that would be two identical
    swatches.
    """
    handles = [plt.Line2D([0], [0], marker=cfg.EDITOR_MARKERS[editor],
                          color="black", label=editor,
                          markerfacecolor=DMS_POINT_COLOR, markersize=4,
                          linewidth=0)
               for editor in ("ABE", "CBE")]
    fig_legend = plt.figure(figsize=(6 / 2.54, 4 / 2.54))
    legend = fig_legend.legend(handles=handles, loc="center", frameon=False)
    for text in legend.get_texts():
        text.set_fontproperties(arial_font6)
    fig_legend.tight_layout()
    fig_legend.savefig(f"{path}-legend.svg", format="svg", dpi=1200,
                       transparent=True)
    plt.close(fig_legend)

scored = dms_frame.dropna(subset=["Dox-UT-Z", "DMS-Score-Z"])

correlation_scatterplot_figure(
    scored, "Dox-UT-Z", "DMS-Score-Z",
    palette=EDITOR_COLORS, legend_order=["ABE", "CBE"], hue_col="Editor",
    highlight_col="Editor", highlight_categories=[],
    axis=replace(AXIS, xlim=(-4, 8), ylim=(-4, 6),
                 xticks=[-4, 0, 4, 8], yticks=[-4, -2, 0, 2, 4, 6],
                 title="LOF hyperactivation vs DMS",
                 xlabel="Hyperactivation Z-score", ylabel="DMS Score"),
    style=dms_corr_style,
    output=replace(OUTPUT, figsize=(6 / 2.54, 4 / 2.54), save=True, rasterize=True,
                   path=str(cfg.OUT_DMS / "All-LOF-Hyperactivation-DMS-correlation")),
)
write_editor_legend(cfg.OUT_DMS / "All-LOF-Hyperactivation-DMS-correlation")

for gene in ("PTPN11", "KRAS"):
    correlation_scatterplot_figure(
        scored[scored["Gene"] == gene], "Dox-UT-Z", "DMS-Score-Z",
        palette=EDITOR_COLORS, legend_order=["ABE", "CBE"], hue_col="Editor",
        highlight_col="Editor", highlight_categories=[],
        axis=replace(AXIS, xlim=(-6, 8), ylim=(-3, 3),
                     xticks=[-6, -4, -2, 0, 2, 4, 6, 8], yticks=[-3, 0, 3],
                     title=f"LOF hyperactivation vs DMS, {gene}",
                     xlabel="Hyperactivation Z-score", ylabel="DMS Score"),
        style=dms_corr_style,
        output=replace(OUTPUT, figsize=(6 / 2.54, 4 / 2.54), save=True, rasterize=True,
                       path=str(cfg.OUT_DMS / f"{gene}-LOF-Hyperactivation-DMS-correlation")),
    )
    write_editor_legend(cfg.OUT_DMS / f"{gene}-LOF-Hyperactivation-DMS-correlation")

erk_scored = lof_erk_dms.dropna(subset=[cfg.LOF_ERK_COLUMN, "DMS-Score-Z"])
correlation_scatterplot_figure(
    erk_scored, cfg.LOF_ERK_COLUMN, "DMS-Score-Z",
    palette=EDITOR_COLORS, legend_order=["ABE", "CBE"], hue_col="Editor",
    highlight_col="Editor", highlight_categories=[],
    axis=replace(AXIS, xlim=(-3, 6), ylim=(-3, 4),
                 xticks=[-3, 0, 3, 6], yticks=[-3, -2, -1, 0, 1, 2, 3, 4],
                 title="LOF hyperactivation vs DMS, ERK2",
                 xlabel="Hyperactivation Z-score", ylabel="DMS Score"),
    style=dms_corr_style,
    output=replace(OUTPUT, figsize=(6 / 2.54, 4 / 2.54), save=True, rasterize=True,
                   path=str(cfg.OUT_DMS / "MAPK1-LOF-ERK-Hyperactivation-DMS-correlation")),
)
write_editor_legend(cfg.OUT_DMS / "MAPK1-LOF-ERK-Hyperactivation-DMS-correlation")

In [ ]:
#@title Hit versus non-hit DMS distribution { display-mode: "form" }

np.random.seed(cfg.RANDOM_STATE)

HIT_CUTOFF = 2

for frame, score_col, name in (
    (dms_frame, "Dox-UT-Z", "All-DMS-by-Hyperactivation-hit"),
    (lof_erk_dms, cfg.LOF_ERK_COLUMN, "MAPK1-LOF-ERK-DMS-by-Hyperactivation-hit"),
):
    usable = frame.dropna(subset=[score_col, "DMS-Score-Z"]).copy()
    usable["hit"] = np.where(usable[score_col] > HIT_CUTOFF, "hit", "not hit")
    jitterbox_kdeplot_figure(
        usable.reset_index(drop=True), xcol="hit", ycol="DMS-Score-Z",
        huecol="Gene", color_dict=cfg.COLOR_MAP,
        axis=replace(AXIS, title="MAPK", ylim=(-6, 6),
                     yticks=np.linspace(-6, 6, 5).tolist(),
                     ylabel="DMS Score", xlabel="Hyperactivation hit"),
        output=replace(OUTPUT, figsize=(3.5 / 2.54, 3.5 / 2.54), save=True,
                       path=str(cfg.OUT_DMS / name)),
    )

## 13. cBioPortal lollipops

Three panels from top to bottom: distance from the inhibitor to each residue, validated screen hits as upward stems, and the count of coding
mutations reported in cBioPortal as downward stems.

In [ ]:
#@title Load cBioPortal mutation counts { display-mode: "form" }

cbioportal = {
    gene: lolli.cbioportal_counts(table, cfg.CBIOPORTAL_MUTATION_TYPES)
    for gene, table in lolli.load_cbioportal().items()
}

for gene, counts in cbioportal.items():
    counts.to_csv(cfg.OUT_LOLLIPOP / f"{gene}-cBioPortal-counts.csv", index=False)

pd.DataFrame([
    {"Gene": gene, "positions_mutated": len(counts),
     "total_mutations": int(counts["Count"].sum()),
     "max_at_one_position": int(counts["Count"].max())}
    for gene, counts in cbioportal.items()
])

In [ ]:
#@title Lollipop helper { display-mode: "form" }

valid_lolli_abe = dat.annotate_validation(valid_abe, gof_abe)
valid_lolli_cbe = dat.annotate_validation(valid_cbe, gof_cbe)

def cbioportal_lollipops(distance_tables, out_suffix):
    """One lollipop per gene that has a structure with distances measured."""
    made = []
    for gene, distances in distance_tables.items():
        subset = dat.combine_editors(valid_lolli_abe, valid_lolli_cbe,
                                     gene=gene, keep_muttypes=False)
        subset = dat.drop_unpositioned(subset)
        if subset.empty:
            continue

        stems = []
        for condition in cfg.GOF_DMSO_COLUMNS:
            if condition not in subset.columns:
                continue
            piece = subset[["pos", "Editor", condition, "mutations"]].copy()
            piece = piece.rename(columns={condition: "Score"})
            piece["Category"] = condition
            stems.append(piece)
        stems = pd.concat(stems, ignore_index=True)
        stems.loc[stems["Score"] > cfg.LOLLIPOP_POS_THRESHOLD].to_csv(
            cfg.OUT_LOLLIPOP / f"{gene}-GOF-cBioPortal{out_suffix}-hits.csv", index=False)

        min_score = cfg.LOLLIPOP_MIN_SCORES[gene]
        max_score = cfg.LOLLIPOP_MAX_SCORES[gene]
        brk = cfg.LOLLIPOP_BREAK_SCORES[gene]
        yticks_bot = [min_score, 0] if brk is None else [min_score, brk[1], brk[0], 0]

        _, _, counts = lolli.lollipop_split_figure(
            struct.interpolate_positions(distances), "Position", "Distance",
            stems, "pos", "Score", cfg.LOLLIPOP_POS_THRESHOLD,
            "Category", cfg.GOF_DMSO_COLUMNS, cfg.LOLLIPOP_POS_COLORS,
            (0, max_score), [0, max_score],
            cbioportal[gene], "pos", "Score", cfg.LOLLIPOP_NEG_THRESHOLD,
            "Database", cfg.LOLLIPOP_NEG_COLORS,
            (min_score, 0), yticks_bot,
            style=replace(
                SPLITLOLLIPOPSTYLE, marker_rasterized=True, ybreak=brk,
                distance_yticks=(cfg.LOLLIPOP_LINE_YTICKS_TALL
                                 if gene in cfg.LOLLIPOP_TALL_GENES
                                 else cfg.LOLLIPOP_LINE_YTICKS_SHORT)),
            axis=replace(AXIS, title=f"{gene} GOF vs cBioPortal",
                         ylim=(min_score, max_score), yticks=[min_score, 0, max_score],
                         xlim=(1 - XMAX[gene] * 0.05, XMAX[gene] * 1.05),
                         xticks=[1, XMAX[gene]],
                         ylabel="Z-Score", xlabel="Amino Acid Position", linewidth=0.5),
            domain=replace(DOMAIN, dom_setting=cfg.DOMAINS_LIST[gene], default_alpha=0.25),
            output=replace(FIG_OUTPUT, figsize=(2, 1.5),
                           path=str(cfg.OUT_LOLLIPOP /
                                    f"{gene}-GOF-cBioPortal{out_suffix}-Lollipop")),
        )
        made.append({"Gene": gene, "structure_set": out_suffix.strip("-") or "PDB", **counts})
    return pd.DataFrame(made)

In [ ]:
#@title Lollipops from co-crystal distances { display-mode: "form" }

pdb_distance_by_gene = {gene: pdb_distances[structure]
                        for gene, structure, *_ in cfg.PDB_STRUCTURES}
pdb_lollipop_counts = cbioportal_lollipops(pdb_distance_by_gene, "")
pdb_lollipop_counts

In [ ]:
#@title Lollipops from AlphaFold distances { display-mode: "form" }

af_distance_by_gene = {gene: af_distances[structure]
                       for gene, structure, *_ in cfg.AF_STRUCTURES}
af_lollipop_counts = cbioportal_lollipops(af_distance_by_gene, "-AF")

lollipop_counts = pd.concat([pdb_lollipop_counts, af_lollipop_counts], ignore_index=True)
lollipop_counts.to_csv(cfg.OUT_SUMMARIES / "cbioportal-lollipop-counts.csv", index=False)
lollipop_counts

## 14. Per-gene boxplots

Z-score distribution of every guide targeting each gene, one panel per screen, editor and condition, with the two control categories on the same axis.

In [ ]:
#@title Per-gene boxplots { display-mode: "form" }

box_frames = {
    ("gof", "ABE"): gof_abe, ("gof", "CBE"): gof_cbe,
    ("meki", "ABE"): meki_abe, ("meki", "CBE"): meki_cbe,
    ("lof", "ABE"): lof_abe, ("lof", "CBE"): lof_cbe,
}

box_summaries = []

for (screen, editor), frame in box_frames.items():
    label = cfg.BOXPLOT_SCREEN_LABELS[screen]
    genes = cfg.BOXPLOT_GENES[screen]
    (ymin, ymax), n_ticks = cfg.BOXPLOT_LIMITS[(screen, editor)]
    order = ana.boxplot_order(genes)
    palette = ana.boxplot_palette(genes)

    for condition in ana.boxplot_conditions(screen, frame):
        boxplot_figure(
            frame, "Gene", condition, palette, lab_order=order,
            axis=replace(AXIS, xlabel="Gene", ylabel="Base Editing Z-Score",
                         title=f"{label} {editor} {condition}, By Gene",
                         ylim=(ymin, ymax),
                         yticks=np.linspace(ymin, ymax, n_ticks).tolist()),
            output=replace(FIG_OUTPUT, figsize=cfg.BOXPLOT_FIGSIZES[screen], dpi=1200,
                           path=str(cfg.OUT_BOXPLOT /
                                    f"{label}-[{condition}]-{editor}-byGene-ZAll-boxplot")),
        )
        box_summaries.append(
            ana.boxplot_summary(frame, screen, editor, condition, genes))

# Non-targeting versus essential-splice-site guides, tested within each editor
# in every panel above. The claim the control figure makes is that these two
# populations differ; this is the number behind it.
control_stats = ana.control_pvalues(box_frames)
control_stats.to_csv(cfg.OUT_SUMMARIES / "boxplot-control-pvalues.csv", index=False)

# Figure 1: controls and essential-splice-site guides, both editors on one
# axis, with that test bracketed over each editor's pair.
for screen, column, (ymin, ymax), n_ticks in cfg.BOXPLOT_CONTROL_FIGURES:
    label = cfg.BOXPLOT_SCREEN_LABELS[screen]
    frame = ana.control_boxplot_frame(
        box_frames[(screen, "ABE")], box_frames[(screen, "CBE")], column)
    if frame is None:
        print(f"  {label} control boxplot skipped: "
              f"no {column!r} column in TableS1-ScreenData.xlsx")
        continue

    comparisons, results = ana.control_frame_comparisons(frame, column)
    box.control_boxplot_figure(
        frame, "Gene", column, cfg.BOXPLOT_CONTROL_PALETTE, comparisons,
        lab_order=cfg.BOXPLOT_CONTROL_LABELS,
        axis=replace(AXIS, xlabel="Gene", ylabel="Base Editing Z-Score",
                     title=f"{label} {column}, By Gene",
                     ylim=(ymin, ymax),
                     yticks=np.linspace(ymin, ymax, n_ticks).tolist()),
        bracket=BRACKET,
        output=replace(FIG_OUTPUT, figsize=cfg.BOXPLOT_CONTROL_FIGSIZE, dpi=1200,
                       path=str(cfg.OUT_BOXPLOT /
                                f"{label}-[{column}]-ABE-CBE-byGene-ZAll-boxplot")),
    )
    for result in results:
        print(f"  {label} {result['comparison']}: p = {result['p']:.3g} "
              f"({result['stars']}), n = {result['n_control']} vs "
              f"{result['n_essential']}")

box_summary = pd.concat(box_summaries, ignore_index=True)
box_summary.to_csv(cfg.OUT_SUMMARIES / "boxplot-quartiles.csv", index=False)
print(f"{len(box_summaries)} per-gene boxplots; "
      f"{len(control_stats)} control tests, "
      f"{int((control_stats['p'] < 0.05).sum())} significant at p < 0.05")
control_stats

## 15. Run summary

Everything written by this notebook.

In [ ]:
written = sorted(p for p in cfg.OUTPUTS_DIR.rglob("*") if p.is_file())
by_dir = pd.Series([p.parent.relative_to(cfg.OUTPUTS_DIR).as_posix() for p in written])
by_type = pd.Series([p.suffix for p in written])

print(f"{len(written)} files written to {cfg.OUTPUTS_DIR}\n")
print(by_dir.value_counts().rename("files").to_string())
print()
print(by_type.value_counts().rename("files").to_string())